Requirements 
Bronze 

- Read CSV using Auto Loader 
- Store as Delta 
- Add ingestion timestamp 
- Track source filename 
- Support schema evolution 

In [0]:
from pyspark.sql.functions import current_timestamp, col

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", "/Volumes/associate_assignment/default/raw/schema")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load("/Volumes/associate_assignment/default/raw/input_data/")
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
)

(
    df.writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/associate_assignment/default/raw/checkpoint")
        .trigger(availableNow=True)
        .toTable("associate_assignment.default.bronze_orders")
)